RAG Self-Consistency
LLM의 확률적 특성을 이용해서, 여러번 답변을 생성하고, 그중에 가장 일관된 답변(다수결)을 채택해서 최종응답으로 사용하는 기법이다.

In [5]:
%pip install sentence_transformers scikit-learn -Uqqq

Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

In [7]:
#가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query = None):
    return [
        Document(page_content="파리의 상징은 에펠탑이며, 1889년에 세워졌습니다."),  # 에펠탑 기본 정보
        Document(page_content="파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다."),  # 도시 구조 및 대표 박물관
        Document(page_content="파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.")  # 관광 규모 및 랜드마크
    ]
retrieve_vectordb('파리')

[Document(metadata={}, page_content='파리의 상징은 에펠탑이며, 1889년에 세워졌습니다.'),
 Document(metadata={}, page_content='파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다.'),
 Document(metadata={}, page_content='파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.')]

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', n = 5)  # 한 번 요청으로 응답 5개 생성
prompt = ChatPromptTemplate.from_template('''
아래 주어진 문서를 참고해서 사용자의 [질문]에 대한 여행일정을 작성해주세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
- 답변은 **최종추천일정:**으로 시작하세요.
- 일자별 일정은 한문장으로 요약하세요.
- 불필요한 서술은 생략하고, 핵심일정만 나열하세요.
''')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'
retrieved_docs = retrieve_vectordb(question)
# 문서 본문만 뽑아서 하나의 문자열 context
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

messages = prompt.format_prompt(context = context, question = question).to_messages()
response = llm.generate([messages])
print(response)

generations=[[ChatGeneration(text='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n1일차: 에펠탑을 시작으로 세느강을 따라 산책하며 파리의 도시 역사와 상징을 감상합니다.  \n2일차: 루브르 박물관을 중심으로 파리의 예술과 역사를 둘러봅니다.  \n3일차: 개선문과 주변 명소를 방문하며 파리의 근현대적 상징을 감상하고 여행을 마무리합니다.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n1일차: 에펠탑을 시작으로 세느강을 따라 산책하며 파리의 도시 역사와 상징을 감상합니다.  \n2일차: 루브르 박물관을 중심으로 파리의 예술과 역사를 둘러봅니다.  \n3일차: 개선문과 주변 명소를 방문하며 파리의 근현대적 상징을 감상하고 여행을 마무리합니다.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0609d-4250-7290-b526-fdd55ca6ee03-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1001, 'total_tokens': 1206, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 318}})), ChatGeneration(text='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 세느강을 따라 산책하며 에펠탑과 개

In [9]:
from pprint import pprint
pprint(response.generations[0])

[ChatGeneration(text='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n1일차: 에펠탑을 시작으로 세느강을 따라 산책하며 파리의 도시 역사와 상징을 감상합니다.  \n2일차: 루브르 박물관을 중심으로 파리의 예술과 역사를 둘러봅니다.  \n3일차: 개선문과 주변 명소를 방문하며 파리의 근현대적 상징을 감상하고 여행을 마무리합니다.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n1일차: 에펠탑을 시작으로 세느강을 따라 산책하며 파리의 도시 역사와 상징을 감상합니다.  \n2일차: 루브르 박물관을 중심으로 파리의 예술과 역사를 둘러봅니다.  \n3일차: 개선문과 주변 명소를 방문하며 파리의 근현대적 상징을 감상하고 여행을 마무리합니다.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0609d-4250-7290-b526-fdd55ca6ee03-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1001, 'total_tokens': 1206, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 318}})),
 ChatGeneration(text='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 세느강을 따라 산책하며 에펠탑과 개선문을 방문해 파리의 

In [10]:
candidates = [gen.text for gen in response.generations[0]]
for i, cand in enumerate(candidates):
    print(f"{i+1} : {cand}")

1 : 최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  
1일차: 에펠탑을 시작으로 세느강을 따라 산책하며 파리의 도시 역사와 상징을 감상합니다.  
2일차: 루브르 박물관을 중심으로 파리의 예술과 역사를 둘러봅니다.  
3일차: 개선문과 주변 명소를 방문하며 파리의 근현대적 상징을 감상하고 여행을 마무리합니다.
2 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 세느강을 따라 산책하며 에펠탑과 개선문을 방문해 파리의 대표적인 상징을 둘러봅니다.  
2일차: 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강 주변을 산책합니다.  
3일차: 개선문과 샹젤리제 거리를 다시 둘러보고 에펠탑 야경을 감상하며 파리 여행을 마무리합니다.
3 : 최종추천일정:
- 1일차: 봄·가을 오전에 세느강을 따라 산책하며 파리의 역사와 도시 발전을 살펴보고 루브르 박물관을 관람합니다.
- 2일차: 오전부터 에펠탑을 방문해 1889년 건립된 파리의 상징을 감상하고 주변 세느강을 둘러봅니다.
- 3일차: 관광객이 비교적 적은 이른 아침에 개선문을 방문해 파리의 대표 명소를 둘러본 뒤 도심을 산책합니다.
4 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 파리 역사 지구를 걸으며 세느강과 루브르 박물관을 둘러봅니다.  
2일차: 1889년에 세워진 에펠탑을 방문한 뒤 세느강변에서 파리 전경을 감상합니다.  
3일차: 개선문과 샹젤리제 거리를 관광하며 파리의 대표적인 역사와 도시 풍경을 즐깁니다.
5 : 최종추천일정:  
1일차(봄·가을 추천): 에펠탑을 관람하고 세느강변을 산책하며 1889년 파리 만국박람회와 도시의 역사를 살펴봅니다.  
2일차: 루브르 박물관에서 파리의 예술과 역사를 감상한 뒤 주변 역사 지구를 둘러봅니다.  
3일차: 개선문과 샹젤리제 거리를 방문하며 파리의 상징적인 도시 경관을 감상하고 세느강 야경을 즐깁니다.


### n개의 응답을 하나로 추출하기

In [11]:
from langchain_core.output_parsers import BaseOutputParser
from sentence_transformers import SentenceTransformer
from pydantic import Field, ConfigDict
from sklearn.cluster import KMeans
from collections import Counter
import numpy as np


class RobustSelfConsistencyParser(BaseOutputParser):
    model_config = ConfigDict(
        arbitrary_types_allowed=True
    )

    n_clusters: int = Field(default=2)

    encoder: SentenceTransformer = Field(
        default_factory=lambda: SentenceTransformer(
            'all-MiniLM-L6-v2'
        )
    )

    @property
    def _type(self) -> str:
        return 'robust_self_consistency_parser'

    def parse(self, generations: list[str]) -> str:
        # 1. 임베딩
        embeddings = self.encoder.encode(generations)
        print(embeddings.shape)

        # 2. 클러스터링
        kmeans = KMeans(
            n_clusters=self.n_clusters,
            random_state=42,
            n_init=10
        )

        kmeans.fit(embeddings)

        print(kmeans.labels_)

        # 3. 다수결 투표
        counts = Counter(kmeans.labels_)

        target_label = max(
            counts,
            key=counts.get
        )

        target_indices = np.where(
            kmeans.labels_ == target_label
        )[0]

        print(target_label)
        print(target_indices)

        # 4. 클러스터 중심점과 가장 가까운 답변 선택
        target_centroid = (
            kmeans.cluster_centers_[target_label]
        )

        distances = np.linalg.norm(
            embeddings[target_indices] - target_centroid,
            axis=1
        )

        representative_idx = np.argmin(distances)

        return generations[
            target_indices[representative_idx]
        ]


parser = RobustSelfConsistencyParser()

final_answer = parser.parse(candidates)

print(f'최종 답변: {final_answer}')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)
[0 0 0 0 1]
0
[0 1 2 3]
최종 답변: 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 세느강을 따라 산책하며 에펠탑과 개선문을 방문해 파리의 대표적인 상징을 둘러봅니다.  
2일차: 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강 주변을 산책합니다.  
3일차: 개선문과 샹젤리제 거리를 다시 둘러보고 에펠탑 야경을 감상하며 파리 여행을 마무리합니다.


In [15]:
# 여행 후보 일정을 여러개 생성 후, 클러스터링 기반 Self-Consistency로 최종 답변 생성하는 함수
def travel_planner(question, verbose = False):

    retrieved_docs = retrieve_vectordb(question)
    context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

    # 프롬프트 템플릿을 메시지 형태로 변환
    messages = prompt.format_prompt(context=context, question=question).to_messages()
    response = llm.generate([messages]) # n=5로 5개 답변 생성
    candidates = [gen.text for gen in response.generations[0]] # 후보 텍스트들만 추출

    # 눈으로 후보 확인하고 싶으면 verbose True
    if verbose:
        for i, cand in enumerate(candidates):
            print(f"{i+1} : {cand}")
            print()

    parser = RobustSelfConsistencyParser() # 대표 답변 고르는 파서
    return parser.parse(candidates) # 최종 대표 답변 반환

question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'
response = travel_planner(question, verbose = True)
print(f"최종 답변 : {response}")

1 : 최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄(4~5월) 또는 가을(9~10월)을 추천합니다.  
- 1일차: 세느강을 따라 산책하며 루브르 박물관과 파리의 역사적 중심지를 둘러봅니다.  
- 2일차: 에펠탑을 방문하고 세느강 주변에서 파리의 대표적인 도시 풍경을 감상합니다.  
- 3일차: 개선문과 샹젤리제 거리를 관광하며 파리의 상징적인 명소를 둘러보고 여행을 마무리합니다.

2 : 최종추천일정:  
- **1일차(봄·가을 오전):** 에펠탑에서 파리의 상징과 1889년 건립 역사를 살펴본 뒤 세느강을 따라 산책합니다.  
- **2일차(오전~오후):** 루브르 박물관을 관람하며 파리의 역사와 예술을 체험하고 주변 명소를 둘러봅니다.  
- **3일차(오전~저녁):** 개선문을 방문해 파리의 대표 관광지를 감상한 뒤 시내를 관광하며 여행을 마무리합니다.

3 : 최종추천일정:  
1일차: 세느강을 따라 산책하며 루브르 박물관에서 파리의 역사와 예술을 둘러보고 주변 구시가지를 탐방하세요.  
2일차: 1889년에 세워진 에펠탑을 방문한 뒤 강변에서 파리의 상징적인 풍경을 감상하세요.  
3일차: 개선문과 샹젤리제 거리를 둘러보며 파리의 근현대 역사와 도시 경관을 즐기세요.  
방문 시기: 쾌적한 날씨와 비교적 여유로운 관광을 위해 봄(4~5월)이나 가을(9~10월)을 추천합니다.

4 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 에펠탑을 관람하고 세느강을 따라 산책하며 1889년 파리 만국박람회와 도시의 역사를 살펴보세요.  
2일차: 루브르 박물관에서 파리의 예술과 역사를 감상한 뒤 세느강변과 주변 고전 건축물을 둘러보세요.  
3일차: 개선문을 방문해 파리의 근현대사를 체험하고 샹젤리제 거리를 산책하며 여행을 마무리하세요.

5 : 최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  
1일차: 세느강을 따라 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  
2일차:

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)
[1 0 0 0 1]
0
[1 2 3]
최종 답변 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 에펠탑을 관람하고 세느강을 따라 산책하며 1889년 파리 만국박람회와 도시의 역사를 살펴보세요.  
2일차: 루브르 박물관에서 파리의 예술과 역사를 감상한 뒤 세느강변과 주변 고전 건축물을 둘러보세요.  
3일차: 개선문을 방문해 파리의 근현대사를 체험하고 샹젤리제 거리를 산책하며 여행을 마무리하세요.
